In [1]:
!pip install langchain_community langchainhub chromadb langchain sentence-transformers groq -q

In [2]:
import getpass
import os

# Set a user agent so WebBaseLoader requests identify themselves (avoids a harmless warning)
os.environ["USER_AGENT"] = "CareAI-RAG-Bot/1.0"

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

Enter Groq API Key:  ········


In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=[
    "https://docs.python.org/3/tutorial/errors.html",
    "https://www.w3schools.com/python/python_try_except.asp",
    "https://realpython.com/python-exceptions/",
    "https://docs.python.org/3/library/exceptions.html",
])
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 4 documents


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")

Split into 156 chunks


## Vector store
`persist_directory="./chroma_db"` saves the embeddings to disk so you don't have to re-scrape and re-embed every session. If `./chroma_db` already has data in it, this appends to the existing collection rather than starting fresh.

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Free local embedding model - no API key needed
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Vector store created/updated with {vectorstore._collection.count()} chunks, persisted to ./chroma_db")

C:\Users\hi\AppData\Local\Temp\ipykernel_4020\1187582820.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created/updated with 156 chunks, persisted to ./chroma_db


**Reloading later?** Skip the scrape + embed cells above and just run:
```python
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
```

In [6]:
retriever = vectorstore.as_retriever()

In [7]:
prompt_template = """You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, just say that you don't know. \
Use ten sentences maximum and keep the answer concise with relevant details.\
Give coding examples if needed.
Question: {question}
Context: {context}
Answer:"""

def build_prompt(question, context):
    return prompt_template.format(question=question, context=context)

print("Prompt template ready")

Prompt template ready


In [8]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_rag(question):
    results = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in results])

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",  # replacement for deprecated llama-3.1-8b-instant
        messages=[
            {"role": "system", "content": "You are a helpful coding assistant for students. Use the provided context to answer clearly. Always include code examples."},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

## General Q&A tests

In [9]:
response = ask_rag("What are the different types of errors in python?")
print(response)

## 1.  Where do “errors” show up in Python?

| Phase | What it is | Typical name | How it’s raised |
|-------|------------|--------------|-----------------|
| **Parsing / compilation** | The interpreter can’t understand the text you wrote. | **SyntaxError** (and its subclasses such as `IndentationError`, `TokenError`) | Immediately, on the first line that contains the problem. |
| **Execution** | The program runs, but something goes wrong at runtime. | **Exceptions** (all subclasses of `BaseException`) | When a statement or function can’t be performed (e.g. dividing by zero, accessing a missing key). |
| **Logic** | The program runs without crashing, but does **not** do what you intended. | *Not an exception at all* | Detected by tests or by manually inspecting output. |

> **Quick recap** – *Syntax errors* prevent the code from even starting.  
> *Exceptions* happen **while** the program is running.  
> *Logical errors* are bugs that produce wrong results but don’t raise an exception.

In [10]:
response = ask_rag("How do I fix an IndexError in Python?")
print(response)

## Quick answer

An **`IndexError`** happens when you try to access an element that doesn’t exist in a sequence (list, tuple, string, etc.).  
To fix it you either

| What you did wrong | Fix |
|---------------------|-----|
| Used an index that is *outside* the valid range (`0` to `len(seq)-1`) | **Check the index** before you use it (`if idx < len(seq): …`) |
| Accidentally used a negative index that is too far back | **Use a positive index** or a safe negative (`-1` for last element, `-len(seq)` for first) |
| Mistyped the variable name that holds the index | **Verify the variable** and its value (`print(idx)` or use a debugger) |
| Assumed a sequence is longer than it actually is (e.g., after filtering) | **Re‑evaluate the length** after any modification (`len(new_seq)`) |

Below are some common patterns that trigger the error and how to correct them.

---

## 1. Accessing an out‑of‑range element

```python
numbers = [10, 20, 30]
print(numbers[3])  # IndexError: list index out of ra

In [11]:
buggy_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    average = total / len(numbers)
    return average

scores = [85, 92, 78, 90, 88]
result = calculate_average(scores)
print("Average score:" + result)
"""

response = ask_rag(f"Find and fix the bug in this code:\n\n{buggy_code}")
print(response)

### What’s wrong?

```python
print("Average score:" + result)
```

`result` is a **float** (the average you calculated), so trying to concatenate it directly with a string raises

```
TypeError: can only concatenate str (not "float") to str
```

In addition, `calculate_average` will throw a `ZeroDivisionError` if you pass an empty list. It’s good practice to guard against that as well.

---

## Fixed version

```python
def calculate_average(numbers):
    """Return the average of a list of numbers.
    
    If the list is empty, return None (or raise a ValueError if you prefer)."""
    if not numbers:                     # guard against division by zero
        # raise ValueError("Cannot calculate average of an empty list")
        return None

    total = 0
    for num in numbers:
        total += num

    average = total / len(numbers)
    return average


scores = [85, 92, 78, 90, 88]
result = calculate_average(scores)

if result is not None:
    # Option 1 – string conversion
    pr